# Exp02: Augmentation Experiments

## 실험 목적
- HNM 데이터셋 기반으로 증강 기법별 성능 영향 확인
- Exp01 결과 (HNM 적용 Baseline)와 비교

## 실험 구성
| 실험 | 파라미터 | 기본값 | 테스트 값 | 비고 |
|---|---|---|---|---|
| Exp02-1 | shear | 0.0 | 5.0, 10.0, 15.0 | 형태 변형 |
| Exp02-2 | mixup | 0.0 | 0.1, 0.2, 0.3 | 두 이미지 블렌딩 |
| Exp02-3 | hsv_h | 0.015 | 0.05, 0.15, 0.25 | 색상 조정 |
| Exp02-4 | hsv_s | 0.7 | 0.5, 0.7, 1.0 | 채도 조정 |
| Exp02-5 | hsv_v | 0.4 | 0.3, 0.6 | 밝기 조정 |
| Exp02-6 | translate | 0.1 | 0.0, 0.05, 0.1 | 번역 감소 |

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
import matplotlib.pyplot as plt

# GPU 확인
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 프로젝트 루트 경로 (Google Drive)
PROJECT_ROOT = Path("/content/drive/MyDrive")
print(f"Project Root: {PROJECT_ROOT}")

Device: cuda
GPU: NVIDIA L4
Project Root: /content/drive/MyDrive


## 공통 설정

In [7]:
from ultralytics import YOLO

# best.pt 경로 (절대 경로 사용)
BEST_PT = PROJECT_ROOT / "runs" / "detect" / "runs" / "train_full" / "yolo11n" / "weights" / "best.pt"
print(f"Base model: {BEST_PT}")
print(f"Exists: {BEST_PT.exists()}")

# 데이터 경로 (data.yaml 파일 경로)
DATA_YAML = str(PROJECT_ROOT / "AI" / "data" / "hnm_dataset" / "data.yaml")
print(f"\nData YAML: {DATA_YAML}")

# 공통 학습 파라미터
EPOCHS = 50
BATCH = 64
IMG_SIZE = 640
PATIENCE = 15

print(f"\n=== Training Configuration ===")
print(f"Epochs: {EPOCHS}")
print(f"Batch: {BATCH}")
print(f"Image Size: {IMG_SIZE}")
print(f"Patience: {PATIENCE}")

Base model: /content/drive/MyDrive/runs/detect/runs/train_full/yolo11n/weights/best.pt
Exists: True

Data YAML: /content/drive/MyDrive/AI/data/hnm_dataset/data.yaml

=== Training Configuration ===
Epochs: 50
Batch: 64
Image Size: 640
Patience: 15


In [8]:
def train_with_augmentation(exp_name, aug_params, epochs=EPOCHS, batch=BATCH):
    """증강 파라미터를 적용하여 학습하고 결과 반환"""
    model = YOLO(str(BEST_PT))

    # 기본 학습 파라미터 + 증강 파라미터 결합
    train_params = {
        "data": DATA_YAML,
        "epochs": epochs,
        "batch": batch,
        "imgsz": IMG_SIZE,
        "device": DEVICE,
        "patience": PATIENCE,
        "workers": 4,
        "amp": False,
        "project": "runs/exp02",
        "name": exp_name,
        "exist_ok": True,
        "pretrained": True,
        "optimizer": "AdamW",
        "cos_lr": True,
        "warmup_epochs": 3,
    }
    train_params.update(aug_params)

    print(f"\n{'='*50}")
    print(f"Training: {exp_name}")
    print(f"Augmentation: {aug_params}")
    print(f"{'='*50}")

    results = model.train(**train_params)
    return results


def print_val_results(results, exp_name):
    """검증 결과 출력"""
    print(f"\n--- {exp_name} Results ---")
    print(f"Precision: {results.box.p:.4f}")
    print(f"Recall: {results.box.r:.4f}")
    print(f"mAP50: {results.box.map50:.4f}")
    print(f"mAP50-95: {results.box.map:.4f}")
    return {
        "Precision": float(results.box.p),
        "Recall": float(results.box.r),
        "mAP50": float(results.box.map50),
        "mAP50-95": float(results.box.map),
    }

In [9]:
# 결과 저장용 딕셔너리
all_results = []

# Exp01 Baseline (HNM only) 결과 - 사전 기록된 값 사용
# 실제 Exp01 실행 후 업데이트 필요
EXP01_BASELINE = {
    "Exp": "Exp01",
    "Parameter": "HNM only",
    "Value": "-",
    "Precision": 0.8121,  # Exp01 실행 후 업데이트
    "Recall": 0.8164,
    "mAP50": 0.8719,
    "mAP50-95": 0.8066,
}
all_results.append(EXP01_BASELINE)
print("Exp01 Baseline loaded:")
print(f"  mAP50: {EXP01_BASELINE['mAP50']:.4f}")
print(f"  mAP50-95: {EXP01_BASELINE['mAP50-95']:.4f}")

Exp01 Baseline loaded:
  mAP50: 0.8719
  mAP50-95: 0.8066


---

## Exp02-1: shear (형태 변형)

- 기본값: 0.0
- 테스트 값: 5.0, 10.0, 15.0

In [ ]:
# Exp02-1: shear 실험
shear_values = [5.0, 10.0, 15.0]

for shear in shear_values:
    exp_name = f"shear_{shear}"
    aug_params = {"shear": shear}

    results = train_with_augmentation(exp_name, aug_params)
    result = print_val_results(results, exp_name)

    all_results.append({
        "Exp": "Exp02-1",
        "Parameter": "shear",
        "Value": shear,
        **result
    })

---

## Exp02-2: mixup (이미지 블렌딩)

- 기본값: 0.0
- 테스트 값: 0.1, 0.2, 0.3

In [ ]:
# Exp02-2: mixup 실험
mixup_values = [0.1, 0.2, 0.3]

for mixup in mixup_values:
    exp_name = f"mixup_{mixup}"
    aug_params = {"mixup": mixup}

    results = train_with_augmentation(exp_name, aug_params)
    result = print_val_results(results, exp_name)

    all_results.append({
        "Exp": "Exp02-2",
        "Parameter": "mixup",
        "Value": mixup,
        **result
    })

---

## Exp02-3: hsv_h (색상 조정)

- 기본값: 0.015
- 테스트 값: 0.05, 0.15, 0.25

In [ ]:
# Exp02-3: hsv_h 실험
hsv_h_values = [0.05, 0.15, 0.25]

for hsv_h in hsv_h_values:
    exp_name = f"hsv_h_{hsv_h}"
    aug_params = {"hsv_h": hsv_h}

    results = train_with_augmentation(exp_name, aug_params)
    result = print_val_results(results, exp_name)

    all_results.append({
        "Exp": "Exp02-3",
        "Parameter": "hsv_h",
        "Value": hsv_h,
        **result
    })

---

## Exp02-4: hsv_s (채도 조정)

- 기본값: 0.7
- 테스트 값: 0.5, 0.7, 1.0

In [ ]:
# Exp02-4: hsv_s 실험
hsv_s_values = [0.5, 0.7, 1.0]

for hsv_s in hsv_s_values:
    exp_name = f"hsv_s_{hsv_s}"
    aug_params = {"hsv_s": hsv_s}

    results = train_with_augmentation(exp_name, aug_params)
    result = print_val_results(results, exp_name)

    all_results.append({
        "Exp": "Exp02-4",
        "Parameter": "hsv_s",
        "Value": hsv_s,
        **result
    })

---

## Exp02-5: hsv_v (밝기 조정)

- 기본값: 0.4
- 테스트 값: 0.3, 0.6

In [ ]:
# Exp02-5: hsv_v 실험
hsv_v_values = [0.3, 0.6]

for hsv_v in hsv_v_values:
    exp_name = f"hsv_v_{hsv_v}"
    aug_params = {"hsv_v": hsv_v}

    results = train_with_augmentation(exp_name, aug_params)
    result = print_val_results(results, exp_name)

    all_results.append({
        "Exp": "Exp02-5",
        "Parameter": "hsv_v",
        "Value": hsv_v,
        **result
    })

---

## Exp02-6: translate (번역 감소)

- 기본값: 0.1
- 테스트 값: 0.0, 0.05, 0.1

In [ ]:
# Exp02-6: translate 실험
translate_values = [0.0, 0.05, 0.1]

for translate in translate_values:
    exp_name = f"translate_{translate}"
    aug_params = {"translate": translate}

    results = train_with_augmentation(exp_name, aug_params)
    result = print_val_results(results, exp_name)

    all_results.append({
        "Exp": "Exp02-6",
        "Parameter": "translate",
        "Value": translate,
        **result
    })

---

## 결과 비교

In [ ]:
# 결과 DataFrame 생성
df = pd.DataFrame(all_results)

# Baseline 대비 개선율 계산
baseline_mAP50 = EXP01_BASELINE["mAP50"]
df["mAP50_diff"] = df["mAP50"] - baseline_mAP50
df["mAP50_diff_pct"] = df["mAP50_diff"] / baseline_mAP50 * 100

print("=== Exp02 Results Summary ===")
print(f"\nBaseline (Exp01): mAP50 = {baseline_mAP50:.4f}")
print()

# 전체 결과 테이블
print(df[["Exp", "Parameter", "Value", "mAP50", "mAP50-95", "mAP50_diff", "mAP50_diff_pct"]].to_string(index=False))

# 최적 결과 찾기
best_idx = df["mAP50"].idxmax()
best_row = df.iloc[best_idx]
print(f"\n{'='*50}")
print(f"Best: {best_row['Exp']} ({best_row['Parameter']}={best_row['Value']})")
print(f"  mAP50: {best_row['mAP50']:.4f} (diff: {best_row['mAP50_diff']:+.4f}, {best_row['mAP50_diff_pct']:+.2f}%)")

In [ ]:
# Exp02 실험만 필터링 (Baseline 제외)
df_exp02 = df[df["Exp"] != "Exp01"].copy()

# 파라미터별 최적 결과
print("=== Parameter-wise Best Results ===")
print()

for param in ["shear", "mixup", "hsv_h", "hsv_s", "hsv_v", "translate"]:
    param_df = df_exp02[df_exp02["Parameter"] == param]
    if len(param_df) > 0:
        best = param_df.loc[param_df["mAP50"].idxmax()]
        diff = best["mAP50"] - baseline_mAP50
        sign = "+" if diff >= 0 else ""
        print(f"{param:12} | Best: {best['Value']:>6} | mAP50: {best['mAP50']:.4f} | {sign}{diff:.4f}")

## 시각화

In [ ]:
# 파라미터별 최적 mAP50 비교 (막대 차트)
param_best = []
for param in ["shear", "mixup", "hsv_h", "hsv_s", "hsv_v", "translate"]:
    param_df = df_exp02[df_exp02["Parameter"] == param]
    if len(param_df) > 0:
        best = param_df.loc[param_df["mAP50"].idxmax()]
        param_best.append({
            "Parameter": param,
            "Best_Value": best["Value"],
            "mAP50": best["mAP50"]
        })

df_param_best = pd.DataFrame(param_best)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. 파라미터별 최적 mAP50
ax1 = axes[0]
bars = ax1.bar(df_param_best["Parameter"], df_param_best["mAP50"], color="steelblue")
ax1.axhline(y=baseline_mAP50, color="red", linestyle="--", label=f"Baseline (Exp01): {baseline_mAP50:.4f}")
ax1.set_xlabel("Parameter")
ax1.set_ylabel("mAP50")
ax1.set_title("Best mAP50 by Parameter")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# 값 표시
for bar, val in zip(bars, df_param_best["Best_Value"]):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{val}', ha="center", va="bottom", fontsize=9)

# 2. Baseline 대비 개선율
ax2 = axes[1]
df_exp02["mAP50_diff_pct"] = (df_exp02["mAP50"] - baseline_mAP50) / baseline_mAP50 * 100
colors = ["green" if x >= 0 else "red" for x in df_exp02["mAP50_diff_pct"]]
ax2.bar(range(len(df_exp02)), df_exp02["mAP50_diff_pct"], color=colors)
ax2.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
ax2.set_xlabel("Experiment Index")
ax2.set_ylabel("mAP50 Improvement (%)")
ax2.set_title("Improvement over Baseline")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 파라미터 값별 성능 변화 (선 그래프)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

params = ["shear", "mixup", "hsv_h", "hsv_s", "hsv_v", "translate"]
for idx, param in enumerate(params):
    ax = axes[idx]
    param_df = df_exp02[df_exp02["Parameter"] == param].sort_values("Value")

    if len(param_df) > 0:
        ax.plot(param_df["Value"], param_df["mAP50"], "o-", color="steelblue", label="mAP50")
        ax.axhline(y=baseline_mAP50, color="red", linestyle="--", alpha=0.7, label="Baseline")
        ax.set_xlabel(param)
        ax.set_ylabel("mAP50")
        ax.set_title(f"{param} Effect")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 결과 CSV 저장
output_csv = PROJECT_ROOT / "AI" / "docs" / "images" / "exp02_results.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"Results saved to: {output_csv}")